# IE University AI Agent Tournament

This notebook is the judge. You use the same one to practise and on tournament day.

| Challenge | Your file | The judge calls | You answer |
|---|---|---|---|
| European Road Trip | `roadtrip_agent.py` | `solve(problem, submit)` | `submit(route)`, as often as you like |
| Chess | `chess_agent.py` | `act(observation)` | `{"move": "e2e4"}` |
| Battleship | `battleship_agent.py` | `act(observation)` | `{"placements": None}` once, then `{"shot": [4, 7]}` each turn |
| Catan | `catan_agent.py` | `act(observation)` | `{"action": 12}` |

`docs/students/CHALLENGES.md`, inside the zip, explains the four challenges in one page.

**To start in Google Colab**
1. Open the Files panel on the left and upload `ai_agent_tournament.zip`.
2. Run the cells in order. Two example teams play a full match, so you can see what happens. A match needs no intervention and takes about 1 minute at most. Keep this tab open while it runs.
3. To play your own team, zip its folder, upload the zip in the Files panel and write its name in section 2. Colab uploads files, not folders. To practise one game with one `.py`, use section 5.

Colab deletes uploaded files when the session ends, so keep your own copies.

It also runs on your own computer: put this notebook next to the zip and run it with Python 3.10 or newer.

## 1. Prepare the environment
Run once per session. Installing does not count against anyone's clock.

In [ ]:
import os, subprocess, sys, zipfile
from pathlib import Path

PACKAGE_ZIP = "ai_agent_tournament.zip"  # change this if you uploaded it somewhere else

WORK_DIR = Path("/content") if Path("/content").is_dir() else Path.cwd()
ROOT = WORK_DIR / "ai_agent_tournament"

def package_is_at(folder):
    return (folder / "tournament" / "__init__.py").is_file()

if package_is_at(Path.cwd()):
    ROOT = Path.cwd()
elif not package_is_at(ROOT):
    archive = next((p for p in [Path(PACKAGE_ZIP), WORK_DIR / PACKAGE_ZIP] if p.is_file()), None)
    if archive is None:
        try:
            from google.colab import files
            print(f"Please choose {PACKAGE_ZIP}")
            archive = Path(next(iter(files.upload())))
        except ImportError:
            raise FileNotFoundError(f"Put {PACKAGE_ZIP} next to this notebook.") from None
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(ROOT)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
if not (ROOT / ".installed").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    (ROOT / ".installed").touch()

from tournament import TeamError, check_requirements, download_result, list_teams, load_team, run_game, run_match, show_game, show_match, verify_environment
assert verify_environment(), "Fix the problems listed above before playing."


## 2. Choose the two teams
Each one is a `.zip` or a folder holding `roadtrip_agent.py`, `chess_agent.py`, `battleship_agent.py` and `catan_agent.py`. For anything you uploaded, the name is enough.

`"demo_a"` and `"demo_b"` are the bundled example teams, so the notebook works before you have written anything.
If your files are in Google Drive, set `USE_GOOGLE_DRIVE = True` and write the full path, such as `/content/drive/MyDrive/Team_12.zip`. They are copied and Drive is disconnected before any agent runs.

In [ ]:
TEAM_A = "demo_a"   # for example "Team_03.zip"
TEAM_B = "demo_b"   # for example "Team_12.zip"
ROUND = None        # optional label, for example "QUARTERFINAL"
USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    local = {}
    for team in (TEAM_A, TEAM_B):
        if team.startswith("/content/drive/"):
            local[team] = f"/content/teams/{Path(team).name}"
            os.makedirs("/content/teams", exist_ok=True)
            if Path(team).is_dir():
                shutil.copytree(team, local[team], dirs_exist_ok=True)
            else:
                shutil.copy(team, local[team])
    TEAM_A, TEAM_B = local.get(TEAM_A, TEAM_A), local.get(TEAM_B, TEAM_B)
    drive.flush_and_unmount()

for team in (TEAM_A, TEAM_B):
    try:
        print("Ready:", load_team(team).name)
    except TeamError as problem:
        print(f"Not ready for a full match: {problem}")


## 3. Play the match
Road Trip (5 seconds at most, both agents at once), then Chess (about 20 seconds at most), Battleship and Catan (about 10 seconds each at most). Every match draws new random cities: Madrid plus 24 others, all different.

When it ends, the notebook prints the result. Tell the professor the winner of each game and which team advances.

In [ ]:
try:
    result = run_match(team_a=TEAM_A, team_b=TEAM_B, round_name=ROUND)
except TeamError as problem:
    result = None
    print(f"No match played. {problem}")


## 4. See how it went
Maps, boards and replays of each game. They are drawn here, after the match, so they never slow the agents down.

In [ ]:
if result:
    show_match(result)
    # download_result(result)  # uncomment to download the match record as JSON


## 5. Practise and check your work (optional)
Practice needs only the one `.py` file you are working on. Your rival is an example agent unless you choose another one.
Errors appear in full. Whatever your agent prints is in `practice["logs"]["A"]`.
`check_requirements` tests your required functions.

In [ ]:
PRACTICE_GAME = None       # "road_trip", "chess", "battleship" or "catan"
MY_AGENT = "demo_a"        # for example "chess_agent.py" or "Team_03.zip"
OPPONENT = "demo"          # "demo", "demo_b", another .py or another team's zip
CHECK_REQUIREMENTS = False

if PRACTICE_GAME:
    practice = run_game(PRACTICE_GAME, MY_AGENT, opponent=OPPONENT)
    show_game(practice)
    print(practice["logs"]["A"])
if CHECK_REQUIREMENTS:
    check_requirements(MY_AGENT, games=[PRACTICE_GAME] if PRACTICE_GAME else None)
if not PRACTICE_GAME and not CHECK_REQUIREMENTS:
    print("Nothing to practise yet: choose PRACTICE_GAME above and run this cell again.")
